# Hyperparameter Optimisation with Optuna — VAE & TransformerVAE

We use **Optuna's NSGA-II sampler** — a multi-objective evolutionary algorithm — to jointly
optimise reconstruction quality and latent-space structure. This gives us a Pareto front rather
than a single scalar optimum, letting us see the trade-off explicitly.

**Why this matters for our project:** our choice of `latent_dim` and `beta` directly affects the
quality of latent representations and, downstream, the drift scores we feed into the SIRC
compartmental model. A principled search gives us evidence that our hyperparameters are
near-optimal — not just eyeballed.

**Reference:** Akiba, T., Sano, S., Yanase, T., Ohta, T., & Koyama, M. (2019).
*Optuna: A Next-generation Hyperparameter Optimization Framework*. KDD.

**Studies in this notebook:**
1. **Joint latent_dim × beta sweep** (VAE) — the two params that matter most for representation quality
2. **Downstream drift sensitivity** — shows the choice affects epidemiological validity, not just reconstruction
3. **TransformerVAE architecture sweep** — justifies d_model, nhead, num_layers

**Note:** We removed the old DiffAE branch from the cleaned submission because it is not part of the final model comparison.
It has fundamentally different training dynamics — noise prediction MSE, not reconstruction
cross-entropy — so the same objective function does not apply.

In [1]:
import os
if not os.path.exists('models'):
    !git clone https://github.com/sidms24/AML.git
    os.chdir('AML')
!pip install -q -r requirements.txt

Cloning into 'AML'...
remote: Enumerating objects: 577, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 577 (delta 77), reused 4 (delta 4), pack-reused 454 (from 2)
Receiving objects: 100% (577/577), 22.47 MiB | 17.88 MiB/s, done.
Resolving deltas: 100% (347/347), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.7/199.7 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.1/182.1 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 14.7 MB/s eta 0:00:00


## Setup

In [2]:
import torch, numpy as np, pandas as pd
import optuna
from optuna.visualization.matplotlib import (
    plot_optimization_history, plot_param_importances,
    plot_slice, plot_contour
)
import tqdm
import tqdm.auto
tqdm.auto.tqdm = tqdm.tqdm
import joblib, warnings, os
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
from scipy.stats import rankdata
from utils.dataloader import load_data, _collate_mixed
from utils.encoders import dna_one_hot
from utils.param_sweep import run_vae_sweep, run_tvae_sweep
from utils.drift import compute_drift_scores
from utils.inference import extract_latents
from utils.loss import VAE_Loss
from utils.train import VAE_train
from models.vae import VAE
from utils.sweep_config import VAE_FIXED
from scipy.stats import spearmanr, rankdata
import matplotlib.pyplot as plt

from datasets import load_dataset
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True

os.makedirs('results', exist_ok=True)

Device: cuda


In [3]:
# we use H1N1_global for the ablation — same dataset as the main training notebook
h1n1_data = load_data(subtype='H1N1_global', encoder=dna_one_hot, batch_size=512)
h1n1_train, h1n1_test = h1n1_data()

sample_x, _ = next(iter(h1n1_train))
INPUT_DIM = sample_x.shape[1]
SEQ_LENGTH = sample_x.shape[2]
print(f'H1N1 — input_dim={INPUT_DIM}, seq_length={SEQ_LENGTH}, '
      f'train batches={len(h1n1_train)}, test batches={len(h1n1_test)}')

Map (num_proc=4): 100%|██████████| 6245/6245 [00:01<00:00, 4537.56 examples/s]


H1N1 — input_dim=5, seq_length=1759, train batches=49, test batches=13


## Study 1: Joint latent_dim × beta sweep x hidden_dim(VAE)

The two hyperparameters that most affect our downstream task are:

- **`latent_dim`** — controls the capacity of the latent space. Too small → can't capture
  variation between seasons. Too large → curse of dimensionality makes L2 drift distances meaningless.
- **`beta`** (KL weight) — controls the trade-off between reconstruction quality and latent
  space regularity. Too low → unstructured latent space. Too high → posterior collapse where
  the encoder ignores the input.

We search these **jointly** because they interact: a small latent space can tolerate higher
beta (the regularisation pressure is spread across fewer dimensions), while a large latent
space may need lower beta to avoid wasting capacity.

All other hyperparameters are **fixed** to their production values for a fair comparison.
We train for 40 epochs (vs 80 in production) since we care about relative ordering, not
absolute best performance.

In [ ]:
vae_study = run_vae_sweep(h1n1_train, h1n1_test, INPUT_DIM, SEQ_LENGTH, device)
joblib.dump(vae_study, 'results/optuna_study_vae.pkl')

In [5]:
# results table — filter out diverged trials
rows = []
for t in vae_study.trials:
    if t.values[0] == float('inf'):
        continue
    rows.append({
        'trial': t.number,
        'latent_dim': t.params['latent_dim'],
        'beta': round(t.params['beta'], 3),
        'hidden_dim': t.params['hidden_dim'],
        'test_recon': round(t.values[0], 4),
        'silhouette': round(-t.values[1], 4),
        'test_kl': round(t.user_attrs['test_kl'], 4),
        'kl_collapsed': t.user_attrs['kl_collapsed'],
        'epochs': t.user_attrs['epochs_trained'],
        'test_loss':  round(t.params['beta'], 3) * t.user_attrs['test_kl'] + t.values[0],

    })
df_vae = pd.DataFrame(rows).sort_values('test_recon')
df_vae.to_csv('results/optuna_vae_results.csv', index=False)
df_vae

,trial,latent_dim,beta,hidden_dim,test_recon,silhouette,test_kl,kl_collapsed,epochs,test_loss
69,69,64,0.018,128,14.1148,0.0374,149.9787,False,40,16.814422
50,50,64,0.021,128,14.3878,0.0378,140.4429,False,40,17.337136
11,11,32,0.286,128,24.4584,0.0470,38.5211,False,40,35.475390
40,40,16,0.018,128,24.5544,0.0343,54.9265,False,40,25.543089
73,73,16,0.154,128,27.3263,0.0319,36.2052,False,40,32.901869
...,...,...,...,...,...,...,...,...,...,...
2,2,8,0.013,16,1128.9514,-0.0633,41.1754,False,40,1129.486673
23,23,8,0.030,16,1141.0983,-0.0662,37.8939,False,40,1142.235099
39,39,8,1.088,16,1177.7676,-0.0384,22.7822,False,40,1202.554560
59,59,8,0.055,16,1245.6497,-0.0576,35.2390,False,40,1247.587879


In [6]:
# Pareto front — these are the non-dominated trials
print(f'Pareto front: {len(vae_study.best_trials)} trials')
for t in vae_study.best_trials:
    print(f"  #{t.number}: latent_dim={t.params['latent_dim']}, "
          f"beta={t.params['beta']:.3f}, recon={t.values[0]:.4f}, "
          f"sil={-t.values[1]:.4f}, "
          f"hidden_dim={t.params['hidden_dim']}, "
          f"total_loss={round(t.params['beta'], 3) * t.user_attrs['test_kl'] + t.values[0]:.4f}")


Pareto front: 4 trials
  #11: latent_dim=32, beta=0.286, recon=24.4584, sil=0.0470, hidden_dim = 128, total_loss: (35.475390229214625,)
  #50: latent_dim=64, beta=0.021, recon=14.3878, sil=0.0378, hidden_dim = 128, total_loss: (17.337135585030524,)
  #69: latent_dim=64, beta=0.018, recon=14.1148, sil=0.0374, hidden_dim = 128, total_loss: (16.814422268439753,)
  #74: latent_dim=128, beta=0.286, recon=34.2360, sil=0.0478, hidden_dim = 64, total_loss: (49.67423711985214,)


In [7]:
from huggingface_hub import upload_folder

upload_folder(
    folder_path="results",
    repo_id="sidms/AML",
    repo_type="dataset",
    path_in_repo = "sweep_results"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...ults/optuna_study_vae.pkl: 100%|██████████| 42.2kB / 42.2kB            

Processing Files (1 / 1)      : 100%|██████████| 42.2kB / 42.2kB,  211kB/s  
New Data Upload               : 100%|██████████| 42.2kB / 42.2kB,  211kB/s  

  ...ults/optuna_study_vae.pkl: 100%|██████████| 42.2kB / 42.2kB            

Processing Files (1 / 1)      : 100%|██████████| 42.2kB / 42.2kB,  105kB/s  
New Data Upload               : 100%|██████████| 42.2kB / 42.2kB,  105kB/s  
  ...ults/optuna_study_vae.pkl: 100%|██████████| 42.2kB / 42.2kB            


CommitInfo(commit_url='https://huggingface.co/datasets/sidms/AML/commit/4a3782973d6179f3c4a08062dd1ee3f3f116ae6e', commit_message='Upload folder using huggingface_hub', commit_description='', oid='4a3782973d6179f3c4a08062dd1ee3f3f116ae6e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/sidms/AML', endpoint='https://huggingface.co', repo_type='dataset', repo_id='sidms/AML'), pr_revision=None, pr_num=None)

In [4]:
VAE_SEARCH_SPACE_STAGE2 = {
    'latent_dim': [32, 64],
    'beta': (0.01, 0.3),
    'hidden_dim': [128],
}

In [ ]:
vae_study2 = run_vae_sweep(h1n1_train, h1n1_test,
                          INPUT_DIM, SEQ_LENGTH,
                          device, search_space = VAE_SEARCH_SPACE_STAGE2,
                           n_trials = 50)
joblib.dump(vae_study2, 'results/optuna_study_vae2.pkl')

In [7]:
from huggingface_hub import upload_file

upload_file(
    path_or_fileobj="results/optuna_study_vae2.pkl",
    repo_id="sidms/AML",
    repo_type="dataset",
    path_in_repo = "sweep_results/optuna_study_vae2.pkl"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...lts/optuna_study_vae2.pkl: 100%|██████████| 26.8kB / 26.8kB            

Processing Files (1 / 1)      : 100%|██████████| 26.8kB / 26.8kB, 26.9kB/s  
New Data Upload               : 100%|██████████| 26.8kB / 26.8kB, 26.9kB/s  

  ...lts/optuna_study_vae2.pkl: 100%|██████████| 26.8kB / 26.8kB            

  ...lts/optuna_study_vae2.pkl: 100%|██████████| 26.8kB / 26.8kB            

  ...lts/optuna_study_vae2.pkl: 100%|██████████| 26.8kB / 26.8kB            

  ...lts/optuna_study_vae2.pkl: 100%|██████████| 26.8kB / 26.8kB            

  ...lts/optuna_study_vae2.pkl: 100%|██████████| 26.8kB / 26.8kB            

Processing Files (1 / 1)      : 100%|██████████| 26.8kB / 26.8kB, 13.4kB/s  
New Data Upload               : 100%|██████████| 26.8kB / 26.8kB, 13.4kB/s  
  ...lts/optuna_study_vae2.pkl: 100%|██████████| 26.8kB / 26.8kB            

CommitInfo(commit_url='https://huggingface.co/datasets/sidms/AML/commit/06cda8a6cbf6bcffd26d2fe2b33edfd2344ea4f4', commit_message='Upload sweep_results/optuna_study_vae2.pkl with huggingface_hub', commit_description='', oid='06cda8a6cbf6bcffd26d2fe2b33edfd2344ea4f4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/sidms/AML', endpoint='https://huggingface.co', repo_type='dataset', repo_id='sidms/AML'), pr_revision=None, pr_num=None)

In [6]:
# results table — filter out diverged trials
rows = []
for t in vae_study2.trials:
    if t.values[0] == float('inf'):
        continue
    rows.append({
        'trial': t.number,
        'latent_dim': t.params['latent_dim'],
        'beta': round(t.params['beta'], 3),
        'hidden_dim': t.params['hidden_dim'],
        'test_recon': round(t.values[0], 4),
        'silhouette': round(-t.values[1], 4),
        'test_kl': round(t.user_attrs['test_kl'], 4),
        'kl_collapsed': t.user_attrs['kl_collapsed'],
        'epochs': t.user_attrs['epochs_trained'],
        'test_loss':  round(t.params['beta'], 3) * t.user_attrs['test_kl'] + t.values[0],

    })
df_vae2 = pd.DataFrame(rows).sort_values('test_recon')
df_vae2.to_csv('results/optuna_vae_results2.csv', index=False)
df_vae2

,trial,latent_dim,beta,hidden_dim,test_recon,silhouette,test_kl,kl_collapsed,epochs,test_loss
42,42,64,0.010,128,14.4041,0.0317,170.8885,False,40,16.113017
25,25,64,0.013,128,14.4728,0.0347,163.1562,False,40,16.593846
22,22,64,0.013,128,14.7729,0.0295,159.2063,False,40,16.842629
41,41,64,0.043,128,14.8218,0.0431,111.1572,False,40,19.601560
8,8,64,0.020,128,14.8630,0.0341,144.6718,False,40,17.756452
14,14,64,0.024,128,14.8879,0.0383,137.5639,False,40,18.189411
7,7,64,0.035,128,15.0055,0.0431,120.0996,False,40,19.209019
9,9,64,0.012,128,15.0703,0.0292,167.3249,False,40,17.078247
29,29,64,0.050,128,15.1239,0.0403,107.7420,False,40,20.511002
47,47,64,0.028,128,15.2777,0.0398,131.8830,False,40,18.970382


In [8]:
# Pareto front — these are the non-dominated trials
print(f'Pareto front: {len(vae_study2.best_trials)} trials')
for t in vae_study2.best_trials:
    print(f"  #{t.number}: latent_dim={t.params['latent_dim']}, "
          f"beta={t.params['beta']:.3f}, recon={t.values[0]:.4f}, "
          f"sil={-t.values[1]:.4f}, "
          f"hidden_dim={t.params['hidden_dim']}, "
          f"total_loss={round(t.params['beta'], 3) * t.user_attrs['test_kl'] + t.values[0]:.4f}")


Pareto front: 13 trials
  #5: latent_dim=64, beta=0.060, recon=15.7960, sil=0.0462, hidden_dim = 128, total_loss: (22.138213903153773,)
  #7: latent_dim=64, beta=0.035, recon=15.0055, sil=0.0431, hidden_dim = 128, total_loss: (19.20901857736189,)
  #11: latent_dim=64, beta=0.156, recon=19.1036, sil=0.0525, hidden_dim = 128, total_loss: (28.7357209411279,)
  #25: latent_dim=64, beta=0.013, recon=14.4728, sil=0.0347, hidden_dim = 128, total_loss: (16.593846018689952,)
  #28: latent_dim=64, beta=0.120, recon=17.9999, sil=0.0504, hidden_dim = 128, total_loss: (26.268323076429894,)
  #30: latent_dim=64, beta=0.133, recon=17.4984, sil=0.0492, hidden_dim = 128, total_loss: (27.65852439154449,)
  #34: latent_dim=64, beta=0.219, recon=19.9987, sil=0.0542, hidden_dim = 128, total_loss: (33.09135218330915,)
  #38: latent_dim=64, beta=0.154, recon=17.8326, sil=0.0500, hidden_dim = 128, total_loss: (28.36303078681695,)
  #39: latent_dim=64, beta=0.063, recon=15.6462, sil=0.0456, hidden_dim = 128, t